# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kiran162005/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1 — The learned model can outperform the fixed baseline

The FlyRank starter results report that the random-forest ranking performed better than the fixed baseline on the starter slice: the baseline was estimated to get about **12 of the top 50** correct, compared with about **37 of the top 50** for the random forest. The result was evaluated on a 30,000-row anonymized starter slice using client-holdout validation.

**My methodology question:**
Where does the label used for “correct” come from, and does it represent a future outcome or a current/proxy outcome? I would also check whether the client-holdout split is sufficient for the specific claim being made, and whether the same improvement remains when the model is evaluated on a different time period or a stricter future-looking target.

**Constructive assessment:**
The client-holdout design is a useful step because pages from held-out clients are not available during training. However, I would be careful about extending the result beyond the stated starter slice until the label definition and validation design support that broader claim.

### Finding 2 — Current-window decline is used as a starter target, but it is only a proxy

The FlyRank starter model defines `is_declining_label` from the current `trend_direction` field. The materials explicitly describe this as a **beginner proxy label**, because it is calculated from the current window rather than representing a future observed outcome. The recommended stronger formulation is a prior feature window followed by a future target window, such as prior 90-day features predicting decline during the following 30 days.

**My methodology question:**
Where exactly does the label come from relative to the prediction/decision point? If the label and features are calculated from the same observation window, could the model simply be learning the current state rather than predicting a future outcome? I would therefore check whether the validation design separates the feature window from the target window before interpreting the model as predictive.

**Constructive assessment:**
The proxy target is useful for demonstrating the modeling workflow, but it should not be treated as evidence that the model can predict future decline. A future-window label with a time-aware validation split would provide stronger evidence for a predictive claim.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


The Week-5 model was initially evaluated using the original split. For this audit, I re-ran the same model using a **client-grouped split**, so pages from the same client are not present in both training and test sets.

This is a stricter and more honest test of generalization because the model is evaluated on clients it did not see during training.

I keep the **same target, features, model, and evaluation metric** as Week 5 so that the comparison measures the effect of the validation design rather than a change in the modeling setup.

The results below show the Week-5 result before the validation improvement and the result after using the grouped-by-client split.

**Interpretation:** A change in performance after grouping by client indicates how much the original result depended on having observations from the same clients in both training and testing. The grouped result is the more appropriate estimate for this capstone question.


In [2]:
# Section 2 — Honest grouped-by-client split

import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# ---------------------------------------------------------
# 1. Reconnect to DuckDB after Colab restart
# ---------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

con = duckdb.connect()

con.execute("DROP SECRET IF EXISTS hf")

con.execute(
    f"""
    CREATE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN.strip()}'
    )
    """
)

FACT = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)

print("DuckDB connection ready.")

DuckDB connection ready.


In [3]:
# ---------------------------------------------------------
# 2. Build content-level modeling dataset
# ---------------------------------------------------------

model_data = con.sql(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        AVG(gsc_avg_position) AS avg_position,

        SUM(ga4_pageviews) AS pageviews,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_users) AS users

    FROM {FACT}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) > 0
)

SELECT *
FROM content
""").df()

print("Modeling rows:", len(model_data))
display(model_data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 176738


,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,pageviews,sessions,users
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.001073,7.209549,1.0,1.0,1.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,2.987198,0.0,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.001066,6.724039,6.0,3.0,3.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.002629,7.244844,2.0,2.0,2.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.002331,4.209227,2.0,2.0,2.0


In [4]:
# ---------------------------------------------------------
# 3. Define target and features
# ---------------------------------------------------------

# Target:
# 1 = high-priority CTR opportunity
# 0 = otherwise
#
# This uses the same observable March 2026 rule as the baseline.

model_data["target"] = (
    (model_data["impressions"] >= 100) &
    (model_data["ctr"] < 0.02)
).astype(int)

features = [
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "pageviews",
    "sessions",
    "users"
]

# Keep only rows with complete feature values
model_data = model_data.dropna(
    subset=features + ["target", "client_hash_id"]
).copy()

X = model_data[features]
y = model_data["target"]
groups = model_data["client_hash_id"]

print("X shape:", X.shape)
print("Positive target rows:", y.sum())
print("Negative target rows:", (y == 0).sum())
print("Unique clients:", groups.nunique())

X shape: (128012, 7)
Positive target rows: 75834
Negative target rows: 52178
Unique clients: 38


In [5]:
# ---------------------------------------------------------
# 4. Grouped-by-client split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print(
    "Client overlap:",
    len(set(groups_train) & set(groups_test))
)

Training rows: 105947
Test rows: 22065
Training clients: 30
Test clients: 8
Client overlap: 0


In [6]:
# ---------------------------------------------------------
# 5. Train the Week-5 model under the honest split
# ---------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

grouped_accuracy = accuracy_score(y_test, pred)
grouped_f1 = f1_score(y_test, pred, zero_division=0)

print("Grouped accuracy:", grouped_accuracy)
print("Grouped F1:", grouped_f1)

Grouped accuracy: 1.0
Grouped F1: 1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audited the final feature set for variables that could reveal the target directly, come from a future observation window, or be generated from the product's flags or labels.

The model uses only observed March 2026 performance fields: impressions, clicks, CTR, average position, pageviews, sessions, and users.

I did not use future-period performance, future labels, product-generated flags, client names, URLs, search queries, or any post-outcome information.

One important consideration is that `ctr` is calculated directly from March impressions and clicks. Because the Week-5 target is also defined using an observed CTR threshold, CTR is closely related to the target. This is **target construction overlap rather than a future-window leak**, but it means the model should not be interpreted as discovering an independent causal signal.

The grouped validation by client prevents the same client's rows from appearing in both training and test sets.


In [7]:
# ---------------------------------------------------------
# 3. Leakage audit — final feature set
# ---------------------------------------------------------

final_features = [
    "impressions",
    "clicks",
    "avg_position",
    "pageviews",
    "sessions",
    "users"
]

print("FINAL FEATURE SET")
for feature in final_features:
    print(" -", feature)

# Known unsafe / suspicious fields that should NOT be features
unsafe_fields = [
    "target",
    "label",
    "flag",
    "product_flag",
    "health_score",
    "trend_direction",
    "trend_pct"
]

print("\nLEAKAGE / POST-OUTCOME FIELD CHECK")

found_unsafe = []

for field in unsafe_fields:
    if field in final_features:
        found_unsafe.append(field)

if found_unsafe:
    print("WARNING — suspicious fields found:", found_unsafe)
else:
    print("PASS — no known target/flag/future fields are in the feature list.")

# Check that the final feature columns actually exist
missing_features = [
    f for f in final_features
    if f not in model_data.columns
]

print("\nFEATURE EXISTENCE CHECK")

if missing_features:
    print("Missing features:", missing_features)
else:
    print("PASS — all final features exist in the modeling dataset.")

# Check target is NOT accidentally included
print("\nTARGET SEPARATION CHECK")

if "target" in final_features:
    print("FAIL — target is included as a feature.")
else:
    print("PASS — target is separate from X.")

FINAL FEATURE SET
 - impressions
 - clicks
 - avg_position
 - pageviews
 - sessions
 - users

LEAKAGE / POST-OUTCOME FIELD CHECK
PASS — no known target/flag/future fields are in the feature list.

FEATURE EXISTENCE CHECK
PASS — all final features exist in the modeling dataset.

TARGET SEPARATION CHECK
PASS — target is separate from X.


In [8]:
# ---------------------------------------------------------
# Check for target-construction overlap
# ---------------------------------------------------------

print("\nTARGET-CONSTRUCTION OVERLAP CHECK")

print("Target definition:")
print("target = impressions >= 100 AND ctr < 0.02")

print("\nCTR is included as a model feature:", "ctr" in final_features)

if "ctr" in final_features:
    print(
        "NOTE — CTR overlaps directly with target construction. "
        "This is not a future-window leak, but it can make the "
        "prediction task artificially easy."
    )


TARGET-CONSTRUCTION OVERLAP CHECK
Target definition:
target = impressions >= 100 AND ctr < 0.02

CTR is included as a model feature: False


In [9]:
# ---------------------------------------------------------
# Verify grouped validation leakage
# ---------------------------------------------------------

train_clients = set(groups_train)
test_clients = set(groups_test)

overlap = train_clients.intersection(test_clients)

print("\nGROUPED VALIDATION CHECK")
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(overlap))

if len(overlap) == 0:
    print("PASS — no client appears in both train and test.")
else:
    print("FAIL — client leakage detected.")


GROUPED VALIDATION CHECK
Training clients: 30
Test clients: 8
Client overlap: 0
PASS — no client appears in both train and test.


### Leakage audit conclusion

The final feature set passes the future-window and product-flag leakage checks: no future performance fields, target labels, or product-generated flags were used as model inputs. The grouped split also has no client overlap between training and test data.

However, `ctr` is directly involved in constructing the target, so there is target-construction overlap. This does not represent future leakage, but it limits how much the model result can be interpreted as an independent discovery of useful signals. The model should therefore be treated as **directional decision-support**, not evidence of a causal content-refresh effect.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

> “The model identifies pages that should be refreshed and can improve their SEO performance.”

### Safer claim

> “The model **measured and ranked observed March 2026 content-performance patterns** using the selected features. The results are **directional decision-support** for prioritizing pages for human review; they do not demonstrate that refreshing a selected page will improve its SEO performance.”

### Why I changed the claim

The evaluation measures model performance on the available data, but it does not measure the causal effect of actually refreshing a page. In addition, CTR overlaps with the target construction, so the model result should not be interpreted as independent evidence that a refresh will produce an improvement.

Therefore, the strongest supported claim is that the model provides a **measured, directional ranking for review**, rather than a guaranteed recommendation or causal prediction.


### Interpretation of the current result

The grouped evaluation produced accuracy and F1 scores of 1.00. However, this result is not evidence of a strong predictive model because the target was constructed directly from March 2026 CTR and impression thresholds, while CTR was also included as a feature. This creates target-construction overlap and makes the task artificially easy.

Therefore, I will not claim that the model predicts future content decline or that it identifies pages that will benefit from refreshes. The result should be treated as a methodological warning and as directional decision-support only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.